# Fine-tune and calibrate the shopping reranker

Use a GPU runtime in Google Colab. Upload the `esci_reranker_bundle.zip` produced by
`python scripts/create_reranker_bundle.py` from the backend directory.

The model learns from all four ESCI grades. Platt scaling uses separate ESCI query
families and estimates the chance that a pair is judged Exact, Substitute, or Complement.
The official test pairs are held out for the reliability diagram and metrics.


In [ ]:
import torch

assert torch.cuda.is_available(), "Select Runtime → Change runtime type → T4 GPU, then reconnect."
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
from google.colab import files
import hashlib, json, os, shutil, subprocess, sys, zipfile

bundle_name = "esci_reranker_bundle.zip"
if not Path("/content", bundle_name).exists():
    uploaded = files.upload()
    assert bundle_name in uploaded, f"Select {bundle_name}."
    del uploaded

project = Path("/content/esci_reranker")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(Path("/content", bundle_name)) as archive:
    for member in archive.infolist():
        assert (project / member.filename).resolve().is_relative_to(project.resolve())
    archive.extractall(project)
for filename, expected in json.loads((project / "checksums.json").read_text()).items():
    assert hashlib.sha256((project / filename).read_bytes()).hexdigest() == expected, filename
os.chdir(project)
print(json.loads(Path("data/manifest.json").read_text()))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentence-transformers==5.7.0", "transformers==5.23.0", "python-dotenv>=1,<2",
                "matplotlib>=3.8,<4"], check=True)


In [ ]:
output = Path("/content/esci_reranker_export")
if not (output / "esci-reranker/calibration.json").exists():
    subprocess.run([sys.executable, "-u", "-m", "src.semantic.train_reranker",
                    "--data", "data", "--output", str(output), "--device", "cuda"], check=True)
else:
    print("Calibration and model export already exist in this runtime.")


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(output / "reports/reliability.png")))
report = json.loads((output / "reports/calibration_report.json").read_text())
for field in ("base_ndcg_at_10", "fine_tuned_ndcg_at_10", "close_match"):
    print(field, report[field])
for name in ("uncalibrated", "platt"):
    print(name, {metric: report[name][metric] for metric in ("brier", "log_loss", "ece")})


In [ ]:
export = shutil.make_archive("/content/esci_reranker_export", "zip", output)
files.download(export)
